In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras.preprocessing.text import Tokenizer

# Multiple sentences
sentences = [
    "the cat is sitting on the mat",
    "the dog is sitting on the chair",
    "the boy is sitting on the bench",
    "the girl is sitting on the sofa",
    "the bird is flying in the sky",
    "the airplane is flying in the sky",
    "the dog is playing in the park",
    "the children are playing in the garden",
    "the cat is sleeping on the bed",
    "the boy is reading a book"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary:", tokenizer.word_index)
print("Vocabulary Size:", vocab_size)

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

# Convert sentences into sequences
sequences = tokenizer.texts_to_sequences(sentences)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary:", tokenizer.word_index)
print("Sequences:", sequences)
print("Vocabulary Size:", vocab_size)

Vocabulary: {'the': 1, 'is': 2, 'on': 3, 'sitting': 4, 'in': 5, 'cat': 6, 'dog': 7, 'boy': 8, 'flying': 9, 'sky': 10, 'playing': 11, 'mat': 12, 'chair': 13, 'bench': 14, 'girl': 15, 'sofa': 16, 'bird': 17, 'airplane': 18, 'park': 19, 'children': 20, 'are': 21, 'garden': 22, 'sleeping': 23, 'bed': 24, 'reading': 25, 'a': 26, 'book': 27}
Vocabulary Size: 28
Vocabulary: {'the': 1, 'is': 2, 'on': 3, 'sitting': 4, 'in': 5, 'cat': 6, 'dog': 7, 'boy': 8, 'flying': 9, 'sky': 10, 'playing': 11, 'mat': 12, 'chair': 13, 'bench': 14, 'girl': 15, 'sofa': 16, 'bird': 17, 'airplane': 18, 'park': 19, 'children': 20, 'are': 21, 'garden': 22, 'sleeping': 23, 'bed': 24, 'reading': 25, 'a': 26, 'book': 27}
Sequences: [[1, 6, 2, 4, 3, 1, 12], [1, 7, 2, 4, 3, 1, 13], [1, 8, 2, 4, 3, 1, 14], [1, 15, 2, 4, 3, 1, 16], [1, 17, 2, 9, 5, 1, 10], [1, 18, 2, 9, 5, 1, 10], [1, 7, 2, 11, 5, 1, 19], [1, 20, 21, 11, 5, 1, 22], [1, 6, 2, 23, 3, 1, 24], [1, 8, 2, 25, 26, 27]]
Vocabulary Size: 28


In [2]:
# Generate Training Data
window_size = 2

X = []
y = []

for sequence in sequences:

    for i in range(window_size, len(sequence) - window_size):

        context = (
            sequence[i-window_size:i] +
            sequence[i+1:i+window_size+1]
        )

        target = sequence[i]

        X.append(context)
        y.append(target)

X = np.array(X)
y = np.array(y)

print("Input Data Shape:", X.shape)
print("Target Data Shape:", y.shape)

print("Training Input:", X)
print("Training Output:", y)

Input Data Shape: (29, 4)
Target Data Shape: (29,)
Training Input: [[ 1  6  4  3]
 [ 6  2  3  1]
 [ 2  4  1 12]
 [ 1  7  4  3]
 [ 7  2  3  1]
 [ 2  4  1 13]
 [ 1  8  4  3]
 [ 8  2  3  1]
 [ 2  4  1 14]
 [ 1 15  4  3]
 [15  2  3  1]
 [ 2  4  1 16]
 [ 1 17  9  5]
 [17  2  5  1]
 [ 2  9  1 10]
 [ 1 18  9  5]
 [18  2  5  1]
 [ 2  9  1 10]
 [ 1  7 11  5]
 [ 7  2  5  1]
 [ 2 11  1 19]
 [ 1 20 11  5]
 [20 21  5  1]
 [21 11  1 22]
 [ 1  6 23  3]
 [ 6  2  3  1]
 [ 2 23  1 24]
 [ 1  8 25 26]
 [ 8  2 26 27]]
Training Output: [ 2  4  3  2  4  3  2  4  3  2  4  3  2  9  5  2  9  5  2 11  5 21 11  5
  2 23  3  2 25]


In [3]:
# Build and Train the CBOW Model
embedding_dim = 50

model = Sequential([
    tf.keras.Input(shape=(window_size * 2,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    ),

    Lambda(lambda x: tf.reduce_mean(x, axis=1)),

    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 4, 50)               │           1,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda (Lambda)                      │ (None, 50)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 28)                  │           1,428 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,828 (11.05 KB)

 Trainable params: 2,828 (11.05 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Train model
history = model.fit(
    X, y,
    epochs=300,
    verbose=1
)

Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.0345 - loss: 3.3299
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.1379 - loss: 3.3221
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.2414 - loss: 3.3144
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.2759 - loss: 3.3065
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.3448 - loss: 3.2987
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.5172 - loss: 3.2908
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.5517 - loss: 3.2829
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.6207 - loss: 3.2749
Epoch 9/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.6552 - loss: 3.2668
Epoch 10/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.6897 - loss: 3.2587
Epoch 11/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.7241 - loss: 3.2504
Epoch 12/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step - accuracy: 0.7241 -

In [5]:
# Test context words
test_words = ["cat", "is", "on", "the"]

# Convert words into indices
test_input = np.array([
    tokenizer.word_index[word] for word in test_words
])

test_input = test_input.reshape(1, -1)

# Predict
prediction = model.predict(test_input, verbose=0)

# Get top 5 predictions
top_indices = np.argsort(prediction[0])[-5:][::-1]

print("Context Words:", test_words)
print("\nTop 5 Predictions:")

for index in top_indices:
    word = tokenizer.index_word.get(index)
    probability = prediction[0][index]

    print(word, ":", round(float(probability), 4))

Context Words: ['cat', 'is', 'on', 'the']

Top 5 Predictions:
sitting : 0.5589
sleeping : 0.2082
is : 0.1475
on : 0.0225
in : 0.0194
